# AR Cloud Multi-Agent RAG System

This notebook demonstrates a multi-agent RAG system with department routing, ChromaDB, OpenAI, and Langfuse.

## 1. Setup and Imports

In [1]:

from pathlib import Path
import json

from src.config import CHROMA_PERSIST_DIRECTORY
from src.retrieval.chunking import DEPARTMENT_FOLDERS
from src.retrieval.retriever import retrieve_documents
from src.router import route_question
from src.agents.hr_agent import answer_hr_question
from src.agents.tech_agent import answer_tech_question
from src.agents.finance_agent import answer_finance_question
from src.orchestrator import process_question
from src.evaluation import load_test_queries

print("Chroma directory:", CHROMA_PERSIST_DIRECTORY)
print("Index available:", CHROMA_PERSIST_DIRECTORY.is_dir())

Chroma directory: /Users/cristianarielachetoni/code/AI Engeenering/M3/department-rag-router/storage/chroma
Index available: True


## 2. Document Collections and Vector Stores

Each department has an independent ChromaDB collection. Run the indexing command before executing retrieval examples.

In [2]:
for department in DEPARTMENT_FOLDERS:
    print(department, f'{department}_documents')

hr hr_documents
technology technology_documents
finance finance_documents


## 3. Specialized Department Agents

In [3]:

retrieval_question = "How do I request annual leave?"

documents = retrieve_documents(
    retrieval_question,
    department="hr",
    k=3,
)

print("Question:", retrieval_question)
print("Retrieved chunks:", len(documents))

for index, document in enumerate(documents, start=1):
    print(f"\nChunk {index}")
    print("Document ID:", document.metadata["document_id"])
    print("Source:", document.metadata["source"])
    print("Content:", document.page_content)
    print("-" * 60)


Question: How do I request annual leave?
Retrieved chunks: 3

Chunk 1
Document ID: HR-002
Source: data/hr_docs/leave_policy.md
Content: The certificate must be submitted through the portal's confidential
HR upload section. Employees must not send medical documents to
team mailing lists, public channels, or handover documents.

HR reviews the certificate and contacts the employee if additional
administrative information is required. Managers receive only
the information needed to organize staffing and the expected
absence period.

For an absence shorter than three consecutive working days,
a certificate is not required under this policy.

## Medical leave recording

Medical leave is recorded separately from annual vacation.
Managers must not automatically deduct vacation days because
an employee reports an illness.

This document does not establish a universal medical leave
entitlement or payment amount. HR confirms the applicable
conditions under the employee's agreement and relevant r

### Department Agent Examples

The system uses three specialized RAG agents: Human Resources, Technology, and Finance. Each agent retrieves information from its own ChromaDB collection and generates answers grounded in department-specific documents.

The following examples demonstrate the behavior of each specialized agent.

In [4]:

department_examples = [
    (
        "Human Resources",
        answer_hr_question,
        "How do I request annual leave?",
    ),
    (
        "Technology",
        answer_tech_question,
        "How do I report a technical incident?",
    ),
    (
        "Finance",
        answer_finance_question,
        "How do I submit a reimbursement request?",
    ),
]

for department_name, agent, user_question in department_examples:
    result = agent(question=user_question, k=3)

    print(f"\nDepartment: {department_name}")
    print(f"Question: {user_question}")
    print(f"Answer: {result['answer']}")
    print("Sources:")

    for source in result["sources"]:
        print(f"- {source['document_id']}: {source['source']}")

    print("-" * 60)


Department: Human Resources
Question: How do I request annual leave?
Answer: To request annual leave at AR Cloud:

1. **Discuss the planned absence with your manager first**, especially if your dates overlap with a product release or another team member’s approved absence.
2. **Submit the vacation request through the employee portal**.
3. Include:
   - the **first day** of your absence,
   - the **last day** of your absence,
   - the **colleague responsible for urgent handover items**.
4. Submit the request **at least 10 working days before** your first day of vacation.

You can also use the employee portal to check your personal vacation balance.
Sources:
- HR-002: data/hr_docs/leave_policy.md
- HR-002: data/hr_docs/leave_policy.md
- HR-001: data/hr_docs/vacation_policy.md
------------------------------------------------------------

Department: Technology
Question: How do I report a technical incident?
Answer: To report a technical incident, create a recorded Technology support requ

## 4. Router and Orchestrator

In [5]:

routing_question = "How do I recover access to my company account?"

routing_decision = route_question(routing_question)

print("Question:", routing_question)
print("Department:", routing_decision.department)
print("Reason:", routing_decision.reason)

Question: How do I recover access to my company account?
Department: technology
Reason: Recovering access to a company account is an IT/account access support issue.


In [6]:
orchestrator_question = "How do I request an expense reimbursement?"

orchestrator_result = process_question(orchestrator_question)

print("Question:", orchestrator_result["question"])
print("Department:", orchestrator_result["department"])
print("Routing reason:", orchestrator_result["routing_reason"])
print("Answer:", orchestrator_result["answer"])
print("Sources:", orchestrator_result["sources"])

Question: How do I request an expense reimbursement?
Department: finance
Routing reason: The question is about requesting reimbursement for expenses, which is a finance procedure.
Answer: To request an expense reimbursement at AR Cloud:

1. **Use the company’s approved submission process**  
   Prepare your reimbursement request through the approved internal process/system.

2. **Include the required expense details**  
   Your request should identify:
   - Employee name
   - Department
   - Expense date
   - Expense category
   - Amount
   - Currency
   - Business justification

3. **Attach supporting documentation**  
   Include sufficient documentation such as:
   - Receipts
   - Invoices
   - Electronic payment confirmations
   - Other Finance-approved records

   Documentation should show the vendor, transaction date, amount paid, currency, and goods or services purchased.

4. **Explain the business purpose**  
   Clearly describe why the expense was necessary for company business

## 5. Tests and Examples

In [7]:
from pathlib import Path
from src.evaluation import load_test_queries

queries = load_test_queries(Path('test_queries.json'))
print('Available evaluation queries:', len(queries))
for query in queries[:3]:
    print(query['id'], query['question'])

Available evaluation queries: 15
HR-01 How do I request annual vacation leave?
HR-02 How can I request a recurring remote work arrangement?
HR-03 Who is responsible for mandatory employee training?


### Out-of-Scope Handling

The router can classify unrelated questions as `out_of_scope`.
In this case, the orchestrator returns a controlled response
without invoking a department RAG agent.

In [8]:
out_of_scope_result = process_question(
    "Who won the World Cup in 2022?"
)

print("Department:", out_of_scope_result["department"])
print("Answer:", out_of_scope_result["answer"])
print("Sources:", out_of_scope_result["sources"])

Department: out_of_scope
Answer: This assistant can only answer questions about AR Cloud's Human Resources, Technology, and Finance policies and procedures.
Sources: []



### Evaluation Results

The evaluation dataset contains 15 queries covering Human Resources,
Technology, Finance, cross-department scenarios, and out-of-scope requests.

The evaluation pipeline measures:

- **Routing accuracy:** Correct department classification, including
  out-of-scope detection.
- **Retrieval hit rate:** Whether at least one expected document was
  retrieved for applicable queries.
- **Answer quality:** LLM judge scores from 1 to 10, based on the
  retrieved context and generated answer.

The LLM judge runs by default during formal evaluations.

Out-of-scope queries are included in routing accuracy but excluded
from retrieval and answer quality metrics.

The following cells load the latest evaluation report without
making additional OpenAI API calls.


In [9]:
report_path = Path("evaluation/report.json")

with report_path.open(encoding="utf-8") as file:
    report = json.load(file)

metrics = report["metrics"]

def format_percentage(value):
    """Format an optional metric as a percentage."""
    return "N/A" if value is None else f"{value:.2%}"


def format_quality(value):
    """Format an optional quality score."""
    return "N/A" if value is None else f"{value:.2f}/10"


print("Evaluation Summary")
print("-" * 40)
print("Total queries:", metrics["total_queries"])
print("Routing evaluated:", metrics["routing_evaluated"])
print("Routing accuracy:", format_percentage(metrics["routing_accuracy"]))
print("Retrieval evaluated:", metrics["retrieval_evaluated"])
print("Retrieval hit rate:", format_percentage(metrics["retrieval_hit_rate"]))
print("Quality evaluated:", metrics["quality_evaluated"])
print(
    "Average answer quality:",
    format_quality(metrics["average_answer_quality"]),
)

Evaluation Summary
----------------------------------------
Total queries: 15
Routing evaluated: 15
Routing accuracy: 100.00%
Retrieval evaluated: 14
Retrieval hit rate: 100.00%
Quality evaluated: 14
Average answer quality: 9.93/10


In [10]:

print(f"{'Query':<12} {'Routing':<12} {'Retrieval':<12} {'Quality'}")
print("-" * 50)

for result in report["results"]:
    routing = result["routing_correct"]
    retrieval = result["retrieval_hit"]
    quality = result.get("answer_quality_score")

    routing_text = "N/A" if routing is None else str(routing)
    retrieval_text = "N/A" if retrieval is None else str(retrieval)
    quality_text = "N/A" if quality is None else f"{quality}/10"

    print(
        f"{result['id']:<12} "
        f"{routing_text:<12} "
        f"{retrieval_text:<12} "
        f"{quality_text}"
    )


Query        Routing      Retrieval    Quality
--------------------------------------------------
HR-01        True         True         10/10
HR-02        True         True         10/10
HR-03        True         True         10/10
HR-04        True         True         9/10
TECH-01      True         True         10/10
TECH-02      True         True         10/10
TECH-03      True         True         10/10
TECH-04      True         True         10/10
FIN-01       True         True         10/10
FIN-02       True         True         10/10
FIN-03       True         True         10/10
FIN-04       True         True         10/10
EDGE-01      True         True         10/10
EDGE-02      True         True         10/10
EDGE-03      True         N/A          N/A


## 6. Langfuse Observability

The orchestrator records routing, retrieval, and generation traces in Langfuse.

Run the orchestration cells above and inspect the resulting traces in the configured Langfuse project.

### Trace Inspection and LLM Judge Scores

Langfuse provides observability for the multi-agent RAG workflow,
including:

- Department routing and LLM generation.
- Document retrieval from department-specific vector stores.
- Input questions, generated answers, and execution metadata.
- Model usage, latency, and token consumption.
- LLM judge scores associated with evaluation traces.

The LLM judge assigns answer quality scores from 1 to 10.

Formal evaluations enable the judge by default. Regular user
queries can optionally enable quality evaluation.

Out-of-scope questions do not receive an answer quality score
because no department-specific RAG answer is generated.

To inspect execution details and quality scores, open the
configured Langfuse project.